# DocFusion: Level 1 — Document Understanding & EDA\n\n**rihal CodeStacker 2026 ML Challenge**\n\nThis notebook explores the unified dataset (SROIE + Find-It-Again + CORD) to build intuition about:\n- Document images and OCR outputs\n- Common layouts, fields, and value distributions\n- Potential anomalies and forgery patterns

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

sns.set_theme(style="whitegrid", palette="husl")
pd.set_option("display.max_columns", 20)

DATA_ROOT = Path("..") / "dummy_data"  # Change to real data path as needed
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
## 1. Load & Explore the Dataset

In [ ]:
# Load training data
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_records = load_jsonl(DATA_ROOT / "train" / "train.jsonl")
test_records = load_jsonl(DATA_ROOT / "test" / "test.jsonl")

print(f"Training records: {len(train_records)}")
print(f"Test records: {len(test_records)}")

# Convert to DataFrame
train_df = pd.json_normalize(train_records)
test_df = pd.json_normalize(test_records)

# Convert total to float
train_df["fields.total"] = train_df["fields.total"].astype(float)
test_df["fields.total"] = test_df["fields.total"].astype(float)

train_df.head(10)

## 2. Label Distribution & Fraud Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Forgery distribution
forgery_counts = train_df["label.is_forged"].value_counts()
axes[0].bar(["Genuine (0)", "Forged (1)"], forgery_counts.values, color=["#34a853", "#ea4335"])
axes[0].set_title("Forgery Label Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(forgery_counts.values):
    axes[0].text(i, v + 0.2, str(v), ha="center", fontweight="bold")

# Fraud type distribution
fraud_counts = train_df["label.fraud_type"].value_counts()
colors = ["#34a853", "#ea4335", "#fbbc04", "#4285f4"]
axes[1].barh(fraud_counts.index, fraud_counts.values, color=colors[:len(fraud_counts)])
axes[1].set_title("Fraud Type Distribution", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Count")

# Vendor distribution
vendor_counts = train_df["fields.vendor"].value_counts()
axes[2].bar(vendor_counts.index, vendor_counts.values, color=sns.color_palette("husl", len(vendor_counts)))
axes[2].set_title("Vendor Distribution", fontsize=14, fontweight="bold")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 3. Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total amount distribution
axes[0].hist(train_df["fields.total"], bins=15, color="#4285f4", edgecolor="white", alpha=0.8)
axes[0].set_title("Total Amount Distribution (Train)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Total ($)")
axes[0].set_ylabel("Count")

# Total by forgery status
genuine = train_df[train_df["label.is_forged"] == 0]["fields.total"]
forged = train_df[train_df["label.is_forged"] == 1]["fields.total"]
axes[1].hist(genuine, bins=12, alpha=0.7, label="Genuine", color="#34a853", edgecolor="white")
axes[1].hist(forged, bins=12, alpha=0.7, label="Forged", color="#ea4335", edgecolor="white")
axes[1].set_title("Total by Forgery Status", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Total ($)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nGenuine — mean: ${genuine.mean():.2f}, std: ${genuine.std():.2f}")
print(f"Forged  — mean: ${forged.mean():.2f}, std: ${forged.std():.2f}")

## 4. Vendor vs Forgery Cross-Analysis

In [ ]:
# Cross-tab: vendor vs forgery
ct = pd.crosstab(train_df["fields.vendor"], train_df["label.is_forged"], margins=True)
ct.columns = ["Genuine", "Forged", "Total"]
display(ct)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
pivot = pd.crosstab(train_df["fields.vendor"], train_df["label.fraud_type"])
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("Vendor × Fraud Type Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Sample Receipt Images & OCR Output

In [ ]:
# Display sample receipt images (requires dummy images to be generated first)
images_dir = DATA_ROOT / "train" / "images"

if images_dir.exists() and list(images_dir.glob("*.png")):
    fig, axes = plt.subplots(2, 5, figsize=(20, 10))
    for idx, rec in enumerate(train_records[:10]):
        ax = axes[idx // 5][idx % 5]
        img_path = DATA_ROOT / "train" / rec["image_path"]
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)
        label = "FORGED" if rec["label"]["is_forged"] == 1 else "GENUINE"
        color = "red" if rec["label"]["is_forged"] == 1 else "green"
        ax.set_title(f'{rec["id"]} — {label}', color=color, fontweight="bold")
        ax.axis("off")
    plt.suptitle("Sample Receipts (Green=Genuine, Red=Forged)", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No images found. Run: python scripts/generate_dummy_images.py")

## 6. Image Feature Analysis (ELA & Statistics)

In [ ]:
from src.preprocessing import compute_image_features, load_image, error_level_analysis
import cv2

images_dir = DATA_ROOT / "train" / "images"

if images_dir.exists() and list(images_dir.glob("*.png")):
    all_features = []
    for rec in train_records:
        img_path = DATA_ROOT / "train" / rec["image_path"]
        if img_path.exists():
            img = load_image(str(img_path))
            feats = compute_image_features(img)
            feats["id"] = rec["id"]
            feats["is_forged"] = rec["label"]["is_forged"]
            all_features.append(feats)

    feat_df = pd.DataFrame(all_features)
    print(f"Extracted {len(feat_df)} feature vectors with {len(feat_df.columns)} features each")
    display(feat_df.describe().round(3))

    # ELA comparison: genuine vs forged
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    genuine_recs = [r for r in train_records if r["label"]["is_forged"] == 0][:2]
    forged_recs = [r for r in train_records if r["label"]["is_forged"] == 1][:2]

    for idx, (rec, title_prefix) in enumerate(
        [(genuine_recs[0], "GENUINE"), (forged_recs[0], "FORGED")]
    ):
        img_path = DATA_ROOT / "train" / rec["image_path"]
        img = load_image(str(img_path))
        ela = error_level_analysis(img)
        axes[idx][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[idx][0].set_title(f'{title_prefix}: {rec["id"]} — Original')
        axes[idx][0].axis("off")
        axes[idx][1].imshow(cv2.cvtColor(ela, cv2.COLOR_BGR2RGB))
        axes[idx][1].set_title(f'{title_prefix}: {rec["id"]} — ELA')
        axes[idx][1].axis("off")

    plt.suptitle("Error Level Analysis: Genuine vs Forged", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No images found. Run: python scripts/generate_dummy_images.py")

## 7. Key Findings & Next Steps\n\n**Observations from EDA:**\n1. Dataset is balanced (50/50 genuine vs forged in dummy data)\n2. Fraud types: price_change, text_edit, layout_edit — each leaves different visual artifacts\n3. Forged receipts tend to have slightly different ELA signatures\n4. Total amount distributions differ between genuine and forged documents\n\n**Next Steps:**\n- Level 2: Build OCR + regex extraction pipeline for vendor/date/total\n- Level 3: Train anomaly detector using image features (ELA) + text features\n- Level 4: Integrate into DocFusionSolution harness

## 2. Label Distribution & Fraud Types